# Stage 2 — Feature Analysis
**Owner: Person B** | Week 1-2

## Goal
For each SAE feature at layers 6, 9, and 11:
- Retrieve the top-20 activating image patches
- Assign a text label via CLIP auto-labeling
- Compute the Monosemanticity Score (Pach et al. 2025)
- Manually annotate the top-50 features per layer

## Rule
No logic in cells. Call functions from src/features.py and src/visualise.py.

## Prerequisite
Person A must have completed 01_sae_setup.ipynb and Person C must
have built the activation cache before running this notebook.

In [ ]:
import sys; sys.path.append("..")
from src.config import get_config
cfg = get_config()

## 1. Load activation cache

In [ ]:
from src.cache import load_layer, load_image_index

image_index = load_image_index()
image_paths = image_index["paths"].tolist()

# Load layer 11 first (primary layer)
acts_11 = load_layer(layer=11)
print("Activations shape:", acts_11.shape)  # (5000, 201, 768)

## 2. Get top-activating patches for all features (layer 11)

In [ ]:
from src.features import get_top_patches_all_features

# This may take a few minutes — runs encode() over all images
top_patches_11 = get_top_patches_all_features(
    layer=11, activations=acts_11, image_paths=image_paths
)
print(f"Computed top patches for {len(top_patches_11)} features")

## 3. CLIP auto-labeling

In [ ]:
from src.features import load_clip_labeler, build_concept_vocab, label_all_features

clip_model, processor = load_clip_labeler()
vocab = build_concept_vocab()
print(f"Vocabulary size: {len(vocab)}")

labels_11 = label_all_features(top_patches_11, clip_model, processor, vocab)
# Spot check
for feat_idx in list(labels_11.keys())[:5]:
    print(f"  Feature {feat_idx}: {labels_11[feat_idx]}")

## 4. Monosemanticity Scores

In [ ]:
from src.features import compute_ms_all_features

ms_scores_11 = compute_ms_all_features(top_patches_11, clip_model, processor)

import statistics
print(f"Mean MS (layer 11): {statistics.mean(ms_scores_11.values()):.3f}")
print(f"Median MS:          {statistics.median(ms_scores_11.values()):.3f}")

## 5. Visualise: MS histogram and feature gallery

In [ ]:
from src.visualise import plot_monosemanticity_histogram, plot_feature_gallery

plot_monosemanticity_histogram(ms_scores_11, layer=11, save=False)

# Gallery of top-20 most monosemantic features
top20 = sorted(ms_scores_11, key=ms_scores_11.get, reverse=True)[:20]
top20_patches = {f: top_patches_11[f] for f in top20}
top20_labels  = {f: labels_11[f] for f in top20}
top20_ms      = {f: ms_scores_11[f] for f in top20}
plot_feature_gallery(top20_patches, top20_labels, top20_ms, n_features=20, save=False)

## 6. Repeat for layers 6 and 9

Copy cells 2-5 above for layers 6 and 9. Compare MS distributions across layers.

In [ ]:
# TODO: repeat for layers 6 and 9
# acts_6  = load_layer(layer=6)
# acts_9  = load_layer(layer=9)
# top_patches_6  = ...
# top_patches_9  = ...

## 7. Manual annotation of top-50 features

After reviewing the feature gallery, manually assign each feature to a category:
texture / color / part / scene / semantic / unclear

Record annotations in report/notes/ as a CSV (feature_idx, layer, category, notes).
Split ~17 features per person for the top-50 inspection.